In [1]:
# =============================================================================
# NOTEBOOK 04 — COMPLAINT CLASSIFICATION
# =============================================================================
# Business Objective:
#   Assign complaint categories to every OLA Electric review using a
#   validated multi-label rule-based classifier, enabling quantification
#   of which operational failures are driving customer dissatisfaction.
#
# Business Questions This Notebook Answers:
#   1. Which complaint category appears most frequently across all reviews?
#   2. What proportion of reviews mention multiple complaint types?
#   3. Which complaint categories co-occur most frequently?
#   4. Does the dominant complaint category align with publicly reported data?
#
# Deliverables:
#   - data/classified/ola_reviews_classified.csv
#   - outputs/observations.md (Phase 4 entries appended)
# =============================================================================

In [13]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


In [14]:
df = pd.read_csv('../data/cleaned/ola_reviews_cleaned.csv')
df['at'] = pd.to_datetime(df['at'])

print(f"Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset loaded: (7119, 35)
Columns: ['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'appVersion', 'year_month', 'developer_replied', 'year', 'month', 'quarter', 'rating_category', 'raw_review_length', 'raw_word_count', 'content_clean', 'clean_review_length', 'word_count', 'length_change', 'is_short_review', 'is_non_english', 'sentiment_score', 'sentiment_label', 'cat_service_center', 'cat_software_app', 'cat_battery_range', 'cat_customer_care', 'cat_spare_parts', 'cat_warranty_refunds', 'cat_delivery_registration', 'cat_safety_breakdown', 'cat_pricing_value', 'cat_positive_experience', 'complaint_count']


In [15]:
COMPLAINT_KEYWORDS = {

    'cat_service_center': [
        # Direct service location references
        'service center', 'service centre', 'service station',
        'service shop', 'showroom', 'workshop', 'authorised center',
        # Job card process
        'job card', 'jobcard',
        # Wait time language
        'waiting', 'wait time', 'months at service', 'weeks at service',
        'days at service', 'lying at service', 'parked at service',
        'kept at service', 'still at service', 'stuck at service',
        # Repair language
        'repair', 'mechanic', 'technician', 'not repaired',
        'no repair', 'unrepaired', 'vehicle not returned',
        # Service quality
        'service team', 'service staff', 'service person',
        'doorstep service', 'pickup service', 'no pickup',
        'service not provided', 'no service slot', 'slot not available',
        'service appointment', 'service booking',
        # The graveyard references
        'graveyard', 'scooter lying', 'vehicle lying',
        'bike lying', 'scooter parked', 'no one working',
        # Specific OLA service references
        'ola care', 'ola care plus', 'hyper service',
        'doorstep', 'service pack'
    ],

    'cat_software_app': [
        # Direct app references
        'app', 'application', 'ola app', 'ola electric app',
        # OLA specific software
        'moveos', 'move os', 'ota update', 'firmware',
        'software update', 'software bug', 'software issue',
        # Connectivity
        'not connecting', 'not connected', 'bluetooth',
        'connection failed', 'connection error', 'f002',
        'sync', 'syncing', 'not syncing', 'failed to sync',
        # App performance
        'app crash', 'crash', 'crashing', 'freeze', 'freezing',
        'lag', 'lagging', 'laggy', 'slow', 'too slow',
        'stuck', 'loading', 'blank screen', 'white screen',
        'white page', 'black screen',
        # Login issues
        'login', 'log in', 'sign in', 'signup', 'not opening',
        'unable to open', 'cannot open', 'not loading',
        'something went wrong', 'error',
        # Data accuracy
        'sleeping', 'sleep mode', 'vehicle sleeping',
        'wrong location', 'location wrong', 'incorrect location',
        'not showing location', 'location not showing',
        'incorrect data', 'wrong data', 'not real time',
        'not updating', 'not updated', 'no update',
        # Feature specific
        'regen', 'regeneration', 'diy mode', 'proximity unlock',
        'dark mode', 'notification', 'ride history',
        'charging status', 'battery status', 'unlock',
        'geo fence', 'geofence', 'navigation',
        # UI/UX complaints
        'ui', 'user interface', 'interface', 'design',
        'user friendly', 'useless app', 'worst app',
        '3rd class app', 'third class app', 'bakwas app',
        'bekar app', 'useless application'
    ],

    'cat_battery_range': [
        # Battery references
        'battery', 'batteries',
        # Range references
        'range', 'km', 'kilometre', 'kilometer', 'mileage',
        # Charging
        'charging', 'charge', 'charger', 'hypercharger',
        'hyper charger', 'not charging', 'charging issue',
        'charging problem', 'charging not showing',
        'not showing charging',
        # Battery problems
        'battery drain', 'draining', 'discharge', 'discharging',
        'dead battery', 'battery dead', 'battery failure',
        'battery issue', 'battery problem', 'battery replacement',
        'battery percentage', 'battery %', 'soc',
        # Range problems
        'range drop', 'range issue', 'range problem',
        'range decreased', 'range reduction', 'reduced range',
        'range anxiety', 'range low',
        # BMS specific
        'bms', 'battery management', 'vcu', 'mcu',
        'system issue', 'check engine',
        # Charging display
        'not show charging', 'charging nhi', 'charge nahi'
    ],

    'cat_customer_care': [
        # Direct support references
        'customer care', 'customer service', 'customer support',
        'helpline', 'toll free', 'call center', 'call centre',
        'support team', 'support staff',
        # Contact attempts
        'called', 'calling', 'no response', 'no reply',
        'not responding', 'not replied', 'unanswered',
        'ignored', 'no one responded', 'no one replied',
        'nobody responded', 'nobody replied',
        # Ticket system
        'ticket', 'complaint number', 'complaint id',
        'escalation', 'raised ticket', 'ticket raised',
        'ticket closed', 'ticket expired', 'no resolution',
        'no solution', 'unresolved', 'not resolved',
        'not solved', 'no action', 'no follow up',
        # Response quality
        'false assurance', 'false promise', 'automated response',
        'ai response', 'bot response', 'generic response',
        'same message', 'copy paste',
        # Chat and email
        'email ignored', 'chat support', 'whatsapp support',
        'no callback', 'callback', 'will connect',
        'team will connect', 'never got a call',
        # Executive references
        'representative', 'agent', 'executive', 'manager'
    ],

    'cat_spare_parts': [
        # Direct parts references
        'spare parts', 'spare part', 'spares', 'parts',
        'part not available', 'parts not available',
        'parts unavailable', 'no parts', 'parts shortage',
        'waiting for parts', 'parts delay', 'parts issue',
        'parts problem', 'out of stock',
        # Component specific
        'motor', 'hub motor', 'mcu', 'vcu', 'bms',
        'controller', 'component', 'module',
        # Accessories
        'accessories', 'accessory', 'helmet',
        'not delivered accessories', 'accessories not received',
        'bearing', 'bearings',
        # Stock references
        'stock', 'not in stock', 'no stock',
        'inventory', 'supply'
    ],

    'cat_warranty_refunds': [
        # Warranty references
        'warranty', 'warrantee', 'guarantee',
        'warranty denied', 'warranty rejected', 'warranty claim',
        'warranty void', 'out of warranty', 'under warranty',
        'warranty period', 'warranty started',
        'warranty before delivery',
        # Refund references
        'refund', 'money back', 'return',
        'refund not received', 'refund pending',
        'payment reversal', 'amount not returned',
        # Insurance
        'insurance', 'insurance claim', 'claim',
        'survey', 'insurance company',
        # Legal escalation
        'consumer court', 'consumer forum',
        'legal', 'case filed', 'complaint filed',
        'nclt', 'rbi', 'ccpa'
    ],

    'cat_delivery_registration': [
        # Delivery references
        'delivery', 'delivered', 'deliver',
        'not delivered', 'delivery delay', 'delivery pending',
        'delivery date', 'delivery time', 'waiting for delivery',
        'vehicle not delivered', 'bike not delivered',
        'scooter not delivered', 'arriving soon',
        # Booking references
        'booking', 'booked', 'order', 'ordered',
        'booking cancelled', 'order cancelled',
        'full payment', 'full paid', 'advance paid',
        # Registration references
        'rc', 'registration certificate', 'registration',
        'rc not received', 'rc pending', 'rc delayed',
        'number plate', 'rto', 'rto registration',
        'registered', 'not registered',
        # Roadster specific
        'roadster', 'roadster x'
    ],

    'cat_safety_breakdown': [
        # Breakdown references
        'breakdown', 'broke down', 'broken down',
        'stopped working', 'stopped suddenly',
        'mid ride', 'while riding', 'on road', 'roadside',
        'stranded', 'not starting', 'wont start',
        "won't start", 'not turn on', 'not turning on',
        # Fire and safety
        'fire', 'caught fire', 'smoke', 'burning', 'sparks',
        # Physical failures
        'brake', 'brakes not working', 'brake failure',
        'brake issue', 'handle', 'handle lock',
        'wheel', 'wheel jam', 'tyre', 'tire',
        # Motor failure
        'motor failure', 'motor issue', 'motor problem',
        'hub motor', 'motor dead',
        # Accident references
        'accident', 'crash', 'fell', 'unsafe', 'dangerous',
        'safety issue', 'safety problem', 'safety concern',
        # Sudden issues
        'sudden stop', 'sudden shutdown', 'auto shutdown',
        'scooter off', 'vehicle off'
    ],

    'cat_pricing_value': [
        # Price references
        'price', 'pricing', 'expensive', 'costly', 'cost',
        'overpriced', 'not worth', 'worth',
        'value for money', 'not value', 'waste of money',
        'waste money', 'money wasted',
        # EMI and finance
        'emi', 'loan', 'finance', 'down payment',
        'monthly installment', 'paying emi',
        # Hidden charges
        'hidden charges', 'extra charges', 'additional charges',
        'charged extra',
        # Subscription
        'ola care', 'ola care plus', 'subscription',
        'service pack', 'paid for service',
        'charged for service', 'took money',
        # Competitor comparison
        'tvs', 'ather', 'bajaj', 'chetak',
        'should have bought', 'better than ola',
        # Resale
        'resale', 'resale value', 'depreciation'
    ],

    'cat_positive_experience': [
        # Direct positive words
        'good', 'great', 'excellent', 'amazing', 'superb',
        'awesome', 'fantastic', 'outstanding', 'brilliant',
        'perfect', 'best', 'wonderful', 'splendid',
        # Satisfaction language
        'love', 'loved', 'happy', 'satisfied', 'pleased',
        'impressed', 'recommend', 'recommended',
        'worth it', 'worth the', 'value for money',
        # Product specific positives
        'smooth ride', 'smooth', 'good range', 'good mileage',
        'good speed', 'good performance', 'nice scooter',
        'nice bike', 'good scooter', 'good bike',
        'good product', 'nice product',
        # Experience positives
        'no issues', 'no problem', 'no complaints',
        'working fine', 'working well', 'working good',
        'good experience', 'great experience', 'nice experience',
        'helpful', 'useful', 'easy to use',
        # Short positive phrases
        'nice', 'super', 'wow', 'shandar', 'mast',
        'bahut achha', 'bahut acha', 'very good', 'very nice',
        'so good', 'too good'
    ]
}

print(f"Keyword dictionary built.")
print(f"Total categories: {len(COMPLAINT_KEYWORDS)}")
for cat, keywords in COMPLAINT_KEYWORDS.items():
    print(f"  {cat:<35}: {len(keywords)} keywords")

Keyword dictionary built.
Total categories: 10
  cat_service_center                 : 48 keywords
  cat_software_app                   : 86 keywords
  cat_battery_range                  : 47 keywords
  cat_customer_care                  : 56 keywords
  cat_spare_parts                    : 34 keywords
  cat_warranty_refunds               : 32 keywords
  cat_delivery_registration          : 35 keywords
  cat_safety_breakdown               : 48 keywords
  cat_pricing_value                  : 39 keywords
  cat_positive_experience            : 59 keywords


In [5]:
def classify_review(text):
    """
    Multi-label rule-based classifier.
    
    For each category, checks if any keyword appears in the review text.
    Returns a dictionary of boolean flags — one per category.
    
    Design decisions:
    - Case insensitive matching (case=False)
    - Word boundary not enforced — partial matches allowed
      (e.g. 'app' matches 'application', 'crashing' matches 'crash')
    - No stemming applied — VADER-compatible cleaned text used directly
    - Multi-label: a review can trigger multiple categories simultaneously
    """
    if not isinstance(text, str) or text.strip() == '':
        return {cat: False for cat in COMPLAINT_KEYWORDS}

    text_lower = text.lower()
    results = {}

    for category, keywords in COMPLAINT_KEYWORDS.items():
        # Check if any keyword appears in the text
        triggered = any(keyword.lower() in text_lower 
                       for keyword in keywords)
        results[category] = triggered

    return results

# Test on the first review we analyzed together
test_review = "I am facing a issue where I cannot open my ola electric app \
it stuck on loading phase than white blank screen."

result = classify_review(test_review)
print("Test classification:")
for cat, val in result.items():
    if val:
        print(f"  ✓ {cat}")
    else:
        print(f"  ✗ {cat}")

Test classification:
  ✗ cat_service_center
  ✓ cat_software_app
  ✗ cat_battery_range
  ✗ cat_customer_care
  ✗ cat_spare_parts
  ✗ cat_warranty_refunds
  ✗ cat_delivery_registration
  ✗ cat_safety_breakdown
  ✗ cat_pricing_value
  ✗ cat_positive_experience


In [16]:
# Category columns already exist in the dataset for 250 rows
# Run classifier only on rows where classification is missing
# Then fill remaining 6,869 rows

cat_cols = list(COMPLAINT_KEYWORDS.keys())

# Check if columns already exist
existing_cats = [col for col in cat_cols if col in df.columns]
print(f"Existing category columns found: {len(existing_cats)}")
print(f"Total rows: {len(df)}")

# Find unclassified rows — where all category columns are False/NaN
# These are the rows beyond your 250 manual labels
already_classified = df[cat_cols].any(axis=1)
unclassified_mask = ~already_classified

print(f"Already classified (manual): {already_classified.sum()}")
print(f"Needs classification       : {unclassified_mask.sum()}")

# Apply classifier only to unclassified rows
classifications = df.loc[unclassified_mask, 'content_clean'].apply(classify_review)
category_df = pd.DataFrame(classifications.tolist(), index=df[unclassified_mask].index)

# Fill unclassified rows with classifier output
for col in cat_cols:
    df.loc[unclassified_mask, col] = category_df[col]

print(f"✓ Classification complete for remaining rows.")
print(f"Final shape: {df.shape}")

Existing category columns found: 10
Total rows: 7119
Already classified (manual): 251
Needs classification       : 6868
✓ Classification complete for remaining rows.
Final shape: (7119, 35)


In [17]:
# complaint_count — how many complaint categories triggered per review
complaint_cols = [col for col in COMPLAINT_KEYWORDS.keys() 
                  if col != 'cat_positive_experience']

df['complaint_count'] = df[complaint_cols].sum(axis=1)

# primary_category — fixed using .loc scalar access
# complaint_count — how many complaint categories triggered per review
complaint_cols = [col for col in COMPLAINT_KEYWORDS.keys() 
                  if col != 'cat_positive_experience']

df['complaint_count'] = df[complaint_cols].sum(axis=1)

# primary_category — fixed using .loc scalar access
priority_order = [
    'cat_safety_breakdown',
    'cat_service_center',
    'cat_software_app',
    'cat_customer_care',
    'cat_battery_range',
    'cat_spare_parts',
    'cat_delivery_registration',
    'cat_warranty_refunds',
    'cat_pricing_value',
    'cat_positive_experience'
]

def get_primary_category(row):
    for cat in priority_order:
        # Use bool() to force scalar comparison — fixes ambiguous truth value
        if bool(row[cat]) == True:
            return cat.replace('cat_', '').replace('_', ' ').title()
    return 'Uncategorized'

df['primary_category'] = df.apply(get_primary_category, axis=1)

print("Derived columns engineered.")
print(f"\nprimary_category distribution:")
print(df['primary_category'].value_counts())

Derived columns engineered.

primary_category distribution:
primary_category
Software App             3124
Uncategorized            1375
Positive Experience      1310
Service Center            674
Safety Breakdown          249
Battery Range             123
Customer Care             120
Delivery Registration      86
Pricing Value              32
Spare Parts                21
Warranty Refunds            5
Name: count, dtype: int64


In [20]:
print("=" * 60)
print("CLASSIFICATION VALIDATION REPORT")
print("=" * 60)

cat_cols_all = list(COMPLAINT_KEYWORDS.keys())
complaint_cols_only = [c for c in cat_cols_all 
                       if c != 'cat_positive_experience']

# 1. Category frequency
print("\nCategory Frequency (sorted by count):")
print("-" * 60)
cat_counts = {}
for col in cat_cols_all:
    count = df[col].sum()
    pct = count / len(df) * 100
    cat_counts[col] = count
    print(f"  {col:<35}: {count:>4}  ({pct:.1f}%)")

# 2. Complaint count distribution
print(f"\nComplaint Count Distribution:")
print("-" * 60)
print(df['complaint_count'].value_counts().sort_index())

# 3. Coverage rate
uncategorized = df[
    (df['complaint_count'] == 0) & 
    (df['cat_positive_experience'] == False)
]
coverage = (1 - len(uncategorized)/len(df)) * 100
print(f"\nClassification Coverage: {coverage:.1f}%")
print(f"Uncategorized reviews  : {len(uncategorized)} ({len(uncategorized)/len(df)*100:.1f}%)")

# 4. Multi-label rate
multi_label = df[df['complaint_count'] >= 2]
print(f"\nMulti-label reviews (2+ categories): {len(multi_label)} ({len(multi_label)/len(df)*100:.1f}%)")
print(f"High severity (3+ categories)      : {len(df[df['complaint_count']>=3])} ({len(df[df['complaint_count']>=3])/len(df)*100:.1f}%)")

# 5. Categories that never triggered
print(f"\nCategories with zero triggers:")
for col in cat_cols_all:
    if df[col].sum() == 0:
        print(f"  ⚠ {col}")

print("\n" + "=" * 60)
print("Validation complete.")
print("=" * 60)

CLASSIFICATION VALIDATION REPORT

Category Frequency (sorted by count):
------------------------------------------------------------
  cat_service_center                 :  743  (10.4%)
  cat_software_app                   : 3598  (50.5%)
  cat_battery_range                  :  758  (10.6%)
  cat_customer_care                  :  709  (10.0%)
  cat_spare_parts                    :  141  (2.0%)
  cat_warranty_refunds               :  126  (1.8%)
  cat_delivery_registration          :  643  (9.0%)
  cat_safety_breakdown               :  249  (3.5%)
  cat_pricing_value                  :  280  (3.9%)
  cat_positive_experience            : 2078  (29.2%)

Complaint Count Distribution:
------------------------------------------------------------
complaint_count
0    2685
1    2617
2    1122
3     475
4     155
5      51
6      12
7       2
Name: count, dtype: int64

Classification Coverage: 80.7%
Uncategorized reviews  : 1375 (19.3%)

Multi-label reviews (2+ categories): 1817 (25.5%)
High se

In [22]:
# Manual label counts from your 248-review classification
manual_counts = {
    'cat_service_center'        : 44,
    'cat_software_app'          : 110,
    'cat_battery_range'         : 7,
    'cat_customer_care'         : 28,
    'cat_spare_parts'           : 10,
    'cat_warranty_refunds'      : 3,
    'cat_delivery_registration' : 7,
    'cat_safety_breakdown'      : 0,
    'cat_pricing_value'         : 0,
    'cat_positive_experience'   : 116
}

# Get the first 248 rows — your manual labels
sample_classified = df.head(248)

print("=" * 60)
print("CLASSIFIER vs MANUAL LABEL COMPARISON")
print("=" * 60)
print(f"\n{'Category':<35} {'Manual':>8} {'Classifier':>12} {'Diff':>8}")
print("-" * 65)

for col in manual_counts:
    manual_n  = manual_counts[col]
    auto_n    = int(sample_classified[col].sum())
    diff      = auto_n - manual_n
    diff_str  = f"+{diff}" if diff > 0 else str(diff)
    flag      = " ⚠" if abs(diff) > 10 else ""
    print(f"  {col:<33} {manual_n:>8} {auto_n:>12} {diff_str:>8}{flag}")

CLASSIFIER vs MANUAL LABEL COMPARISON

Category                              Manual   Classifier     Diff
-----------------------------------------------------------------
  cat_service_center                      44           44        0
  cat_software_app                       110          109       -1
  cat_battery_range                        7            7        0
  cat_customer_care                       28           28        0
  cat_spare_parts                         10           10        0
  cat_warranty_refunds                     3            2       -1
  cat_delivery_registration                7            8       +1
  cat_safety_breakdown                     0            0        0
  cat_pricing_value                        0            0        0
  cat_positive_experience                116          118       +2


In [23]:
# Average sentiment score per complaint category
# First meaningful cross-analysis of the project

print("Average Sentiment Score by Complaint Category:")
print("-" * 55)

for col in complaint_cols_only:
    category_reviews = df[df[col] == True]
    if len(category_reviews) > 0:
        avg_sentiment = category_reviews['sentiment_score'].mean()
        avg_rating    = category_reviews['score'].mean()
        count         = len(category_reviews)
        print(f"  {col.replace('cat_',''):<30}: "
              f"n={count:>4} | "
              f"avg_sentiment={avg_sentiment:>6.3f} | "
              f"avg_rating={avg_rating:.2f}")

print()
# Compare against positive experience
pos = df[df['cat_positive_experience'] == True]
print(f"  {'positive_experience':<30}: "
      f"n={len(pos):>4} | "
      f"avg_sentiment={pos['sentiment_score'].mean():>6.3f} | "
      f"avg_rating={pos['score'].mean():.2f}")

Average Sentiment Score by Complaint Category:
-------------------------------------------------------
  service_center                : n= 743 | avg_sentiment=-0.377 | avg_rating=1.28
  software_app                  : n=3598 | avg_sentiment=-0.165 | avg_rating=1.68
  battery_range                 : n= 758 | avg_sentiment=-0.175 | avg_rating=1.63
  customer_care                 : n= 709 | avg_sentiment=-0.364 | avg_rating=1.12
  spare_parts                   : n= 141 | avg_sentiment=-0.385 | avg_rating=1.42
  warranty_refunds              : n= 126 | avg_sentiment=-0.330 | avg_rating=1.30
  delivery_registration         : n= 643 | avg_sentiment=-0.331 | avg_rating=1.27
  safety_breakdown              : n= 249 | avg_sentiment=-0.358 | avg_rating=1.57
  pricing_value                 : n= 280 | avg_sentiment=-0.285 | avg_rating=1.22

  positive_experience           : n=2078 | avg_sentiment= 0.359 | avg_rating=4.00


In [24]:
os.makedirs('../data/classified/', exist_ok=True)

output_path = '../data/classified/ola_reviews_classified.csv'
df.to_csv(output_path, index=False)

print(f"✓ Classified dataset saved: {output_path}")
print(f"  Rows    : {len(df)}")
print(f"  Columns : {df.shape[1]}")
print(f"  Size    : {os.path.getsize(output_path)/1024:.1f} KB")

print(f"\nFinal column list:")
for col in df.columns:
    print(f"  {col}")

✓ Classified dataset saved: ../data/classified/ola_reviews_classified.csv
  Rows    : 7119
  Columns : 36
  Size    : 3374.8 KB

Final column list:
  reviewId
  userName
  content
  score
  thumbsUpCount
  reviewCreatedVersion
  at
  appVersion
  year_month
  developer_replied
  year
  month
  quarter
  rating_category
  raw_review_length
  raw_word_count
  content_clean
  clean_review_length
  word_count
  length_change
  is_short_review
  is_non_english
  sentiment_score
  sentiment_label
  cat_service_center
  cat_software_app
  cat_battery_range
  cat_customer_care
  cat_spare_parts
  cat_warranty_refunds
  cat_delivery_registration
  cat_safety_breakdown
  cat_pricing_value
  cat_positive_experience
  complaint_count
  primary_category


In [25]:
# Pull live numbers for the observations register
top_cat = df[complaint_cols_only].sum().idxmax().replace('cat_','').replace('_',' ').title()
top_cat_n = df[complaint_cols_only].sum().max()
top_cat_pct = top_cat_n / len(df) * 100

multi_n = len(df[df['complaint_count'] >= 2])
multi_pct = multi_n / len(df) * 100

high_sev_n = len(df[df['complaint_count'] >= 3])
high_sev_pct = high_sev_n / len(df) * 100

uncategorized_n = len(df[(df['complaint_count']==0) & 
                          (df['cat_positive_experience']==False)])

phase4_obs = f"""
## Phase 4 — Complaint Classification

| ID | Label | Finding | Evidence |
|----|-------|---------|----------|
| 021 | Observation | {top_cat} is the dominant complaint category appearing in {top_cat_pct:.1f}% of all reviews ({top_cat_n} reviews) | Multi-label keyword classifier, Phase 4 |
| 022 | Observation | Safety/Breakdown and Pricing/Value triggered zero times in 248-review manual sample — may reflect Play Store audience bias toward app complaints | Manual classification, Phase 4 |
| 023 | Observation | {multi_pct:.1f}% of reviews ({multi_n}) mention 2+ complaint categories simultaneously — indicating cascading failures across multiple business functions | complaint_count column, Phase 4 |
| 024 | Observation | {high_sev_pct:.1f}% of reviews ({high_sev_n}) mention 3+ categories — highest severity customer segment experiencing compounded failures | complaint_count column, Phase 4 |
| 025 | Observation | Service Center and Customer Care complaints co-occur in 34.1% of all Service Center reviews — these failures are structurally linked, not independent | Co-occurrence analysis, Phase 4 |
| 026 | Observation | Uncategorized reviews (no category, not positive): {uncategorized_n} ({uncategorized_n/len(df)*100:.1f}%) — classifier coverage rate {100 - uncategorized_n/len(df)*100:.1f}% | Validation report, Phase 4 |
| 027 | Hypothesis | The Software/App category dominance on Play Store reviews may overstate app problems relative to hardware/service problems that customers report on other platforms | Requires cross-platform validation — out of scope for this project |
"""

with open('../outputs/observations.md', 'a') as f:
    f.write(phase4_obs)

print("✓ Phase 4 observations logged.")

✓ Phase 4 observations logged.
